# finance / portfolio / withdrawal

**Intent**
- Takes `finance_portfolio_withdrawal_age_years`, an age.
- Outputs `finance_portfolio_withdrawal_rate_pct`, the sustainable withdrawal rate.
- It is the percent of wealth withdrawn each year in real terms, from a risk-free return to the planning horizon.
- It is the ceiling any risky mix has to beat.
- Acceptance holds the rmse on `validation` and `backtest` to 0.01.

**Data**
- Personal data is git-ignored in `data/`, with synthetic placeholders committed.
- `training` is ten ages with the rate, `validation` is five further, and `backtest` is ten across the range.
- `backtest` is held back for acceptance.
- `golden` is ages 30 to 120, then a request with the input missing.

**Status**
- The real return is recovered from synthetic points, a placeholder of 2 percent to age 100.

**Future**
- Overall the work still to do before the first release is below.
  - Add `query_postgres` to `workbench.data` and load the real series in place of the synthetic ones.
  - Use the annuity rate `r / (1 - (1 + r)^-n)`, with `r` from the index linked bond yield.
  - Replace the fixed horizon with the survival curve from ABS life tables, joint with your partner.
  - Add the Age Pension and superannuation preservation as an income floor.
  - Fix the limits against real data, then register and review before any release.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
import json
from os.path import join

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import load_signature, provenance, record_project, scratch_project_dir, signature_names

env = init_env()

WORKSPACE = "finance/portfolio/withdrawal"
MODEL_NAME = "finance_portfolio_withdrawal"
TARGET = {"kind": "served"}
signature = load_signature(WORKSPACE)
INPUTS, OUTPUTS = signature_names(signature)
TRAINING = "portfolio_withdrawal_training_2026-11-05_2026-11-05"
VALIDATION = "portfolio_withdrawal_validation_2026-11-05_2026-11-05"
BACKTEST = "portfolio_withdrawal_backtest_2026-11-05_2026-11-05"
GOLDEN = "portfolio_withdrawal_golden"

## Capture Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-capture-data))

In [ ]:
HORIZON = 100.0


def rate(age, real=0.02):
    return 100.0 * real / (1.0 - (1.0 + real) ** -max(HORIZON - age, 1.0))


def labelled(ages):
    return {INPUTS[0]: ages, OUTPUTS[0]: [round(rate(age), 6) for age in ages]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled([40.0, 45.0, 50.0, 55.0, 60.0, 65.0, 70.0, 75.0, 80.0, 85.0]))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled([42.0, 52.0, 62.0, 72.0, 82.0]))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled([41.0, 47.0, 53.0, 59.0, 67.0, 73.0, 79.0, 83.0, 87.0, 90.0]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-fit))

In [ ]:
from scipy.optimize import curve_fit

fitted_real = curve_fit(
    lambda age, real: 100.0 * real / (1.0 - (1.0 + real) ** -(HORIZON - age)), training_pl[INPUTS[0]].to_numpy(), training_pl[OUTPUTS[0]].to_numpy(), p0=[0.03], bounds=(0.001, 0.2)
)[0][0]
PARAMS = {"real": round(float(fitted_real), 4), "horizon": HORIZON}
print("fitted", PARAMS)
assert PARAMS == {"real": 0.02, "horizon": 100.0}, "the fit did not recover the real return"

## Evaluate ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-evaluate))

In [ ]:
from workbench.model import evaluate_acceptance, measure
from workbench.replay import model_predict

MODEL_TEMPLATE = """import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT = {inputs[0]!r}
OUTPUT = {outputs[0]!r}
REAL = {real!r}
HORIZON = {horizon!r}


class Withdrawal(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        return pd.DataFrame({{OUTPUT: [100.0 * REAL / (1.0 - (1.0 + REAL) ** -max(HORIZON - age, 1.0)) for age in model_input[INPUT]]}})


set_model(Withdrawal())
"""

model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS, inputs=INPUTS, outputs=OUTPUTS))

predict = model_predict(model_path, "pyfunc", signature)

In [ ]:
LIMIT = 0.01
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({INPUTS[0]: frame[INPUTS[0]].to_list()}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMIT, "direction": "max"}
        for split, frame in SPLITS
        for output in OUTPUTS
    ]


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
def golden_samples():
    xs = [30.0, 50.0, 65.0, 80.0, 99.5, 120.0]
    answers = predict({INPUTS[0]: xs})
    return {INPUTS[0]: [*xs, None], **{output: [*answers[output], None] for output in OUTPUTS}, "expect": ["ok"] * len(xs) + ["error"]}

In [ ]:
from workbench.model import package_project, register_preview
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "pyfunc",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic annuity withdrawal rates, training split",
        [training_path, golden_path],
        "least squares real return",
        implementation="mlflow.pyfunc model.py, requires mlflow and pandas",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="from early to past the horizon",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGET)
print("replayed the golden sequences exactly through the shipped runtime")

## Register ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-register))

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGET, metadata["version"])
print("registered version", version)